# Exploración Saber Pro: ¿se puede predecir el puntaje global?

Spike de factibilidad de datos (no es el proyecto final), tercer candidato. Predicción del puntaje global de Saber Pro a partir de variables **anteriores a la prueba** (socioeconómicas, institución y programa).

Cada afirmación se marca como **[VERIFICADO]** (lo vimos en los datos) o **[SUPUESTO]**.

## 1. Disponibilidad de datos

**datos.gov.co (catálogo, 10 búsquedas el 2026-10-15)**: hay un conjunto "Resultados únicos Saber Pro" (`u37r-hjmu`), uno de Saber TyT, uno de Saber 11 (`kgxf-xxbe`) y versiones por periodo antiguas (2019-2, 2020-1/2). El ICFES los publica con licencia **Creative Commons Atribución-CompartirIgual 4.0 (CC BY-SA 4.0)** y atribución al ICFES: hay que citar la fuente y compartir las obras derivadas con la misma licencia.

Años disponibles en Saber Pro (`u37r-hjmu`, descripción: "desde el año 2018 al año 2022", última actualización **2023-08-23**): ver la tabla siguiente. **No hay 2023, 2024 ni 2025** en datos.gov.co.

In [1]:
import time, math, json, io
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import requests
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 90)

RAIZ = Path("data/raw/saberpro")
META = RAIZ / "meta"
CACHE = RAIZ / "paginas"
CACHE.mkdir(parents=True, exist_ok=True)
SEED = 42
BASE = "https://www.datos.gov.co"
DID = "u37r-hjmu"
proc = psutil.Process()


def ram(etiqueta):
    mi = proc.memory_info()
    print(f"[RAM] {etiqueta}: actual {mi.rss / 1e6:.0f} MB | pico {getattr(mi, 'peak_wset', mi.rss) / 1e6:.0f} MB")


ram("inicio")
display(pd.read_csv(META / "datasets_icfes_en_datosgov.csv"))
per = pd.read_csv(META / "saberpro_filas_por_periodo.csv", dtype={"periodo": str})
per["anio"] = per.periodo.str[:4].astype(int)
print("filas totales:", int(per.filas.sum()))
display(per.groupby("anio").filas.sum().to_frame("filas").T)
display(per.set_index("periodo").T)

[RAM] inicio: actual 141 MB | pico 141 MB


,id,nombre,actualizado,columnas
0,u37r-hjmu,Resultados unicos Saber Pro,2023-08-23,57
1,iwgf-bkfk,Resultados unicos Saber TyT,2023-08-23,57
2,kgxf-xxbe,Resultados unicos Saber 11,2023-08-23,51
3,2x55-9wxm,Resultados Saber Pro Competencias Genericas 2019-2,2020-02-28,105
4,6kwm-9788,Resultados Saber Pro Competencias Especificas 2019-2,2020-02-28,5
5,4u2m-pguf,Resultados Saber TyT Genericas 2019-2,2020-02-28,102
6,upcp-qane,Resultados Saber TyT Especificas 2019-2,2020-02-28,4
7,k67a-zkz9,Resultados Saber TyT Genericas 2020-1,2021-03-16,96
8,rnvb-vnyh,Saber 11 2020-2,2021-03-15,81
9,a8xr-en99,Saber 11 2020-1,2021-03-15,78


filas totales: 1217482


anio,2018,2019,2020,2021,2022
filas,235597,260756,246455,248769,225905


periodo,20183,20184,20194,20195,20196,20202,20203,20212,20213,20222,20223,20225,20226
filas,235224,373,2076,258452,228,680,245775,247099,1670,97889,674,126282,1060
anio,2018,2018,2019,2019,2019,2020,2020,2021,2021,2022,2022,2022,2022


**[VERIFICADO]** `u37r-hjmu` tiene 1.217.482 filas, periodos 2018-3 a 2022-6 y 57 columnas. Los periodos grandes (≥ 97 mil filas) son 2018-3, 2019-5, 2020-3, 2021-2, 2022-2 y 2022-5; los demás tienen menos de 2.100 filas (rezagados).

### Portal propio del ICFES (años 2023-2025)

- **Su `robots.txt` dice `User-agent: * / Disallow: /`.** Antes de leerlo ya se había hecho un GET a la página de inicio de DataIcfes y a otra del sitio (sin guardar ni leer su contenido). No se hicieron más consultas.
- **No se pudo verificar** si el portal exige registro. Lo que se encontró en fuentes de terceros (búsquedas web, sin confirmar): el portal **DataIcfes** reemplazó al FTP antiguo (2020) y el FTP antiguo exigía registrarse con correo y contraseña. El procedimiento vigente no se pudo confirmar.
- No se creó ninguna cuenta. Para saber si los años 2023-2025 existen y cómo se piden, hay que abrir el portal a mano o escribir al ICFES.

### Cruce con Saber 11 (valor agregado)

- Ambos conjuntos traen `estu_consecutivo`, pero **no es el mismo identificador**: en Saber Pro es `EK201830011083` y en Saber 11 es `SB11201310000414` (prefijos y estructura distintos; cada código identifica un registro de examen). **[VERIFICADO]** con 4 filas de cada conjunto, leyendo solo esa columna y el periodo.
- Saber Pro solo trae `estu_tipodocumentosb11` (el *tipo* de documento con el que presentó Saber 11), no el número ni el consecutivo de Saber 11.
- Los datos están anonimizados: no hay número de documento. Cruzar por combinaciones de fecha de nacimiento, género y colegio sería **reidentificar personas**; no se hizo ni se recomienda.
- **Conclusión: con estos datos públicos no hay llave para ligar a una misma persona entre Saber 11 y Saber Pro.** El "valor agregado" individual no es viable. (El ICFES describe una base longitudinal que une Saber 11 con educación superior, pero sus condiciones de acceso no se pudieron verificar.)
- Observación: en la muestra de Saber 11 apareció el mismo `estu_consecutivo` dos veces seguidas (duplicados), a revisar si se usara.

## 2. Inventario de columnas (Saber Pro `u37r-hjmu`)

In [2]:
meta = json.loads((META / "u37r-hjmu.json").read_text(encoding="utf-8"))
cols = pd.DataFrame({"columna": [c["fieldName"] for c in meta["columns"]], "tipo": [c["dataTypeName"] for c in meta["columns"]],
                     "descripcion": [(c.get("description") or "") for c in meta["columns"]]})
print("columnas:", len(cols), "| tipos:", cols.tipo.value_counts().to_dict())
print("columnas con 'global' en el nombre:", cols[cols.columna.str.contains("glob")].columna.tolist())


def familia(c):
    if c.startswith("mod_"):
        return "puntajes/desempenos por modulo (OBJETIVO o FUGA)"
    if c.startswith("fami_") or c in {"estu_horassemanatrabaja", "estu_valormatriculauniversidad"} or c.startswith("estu_pagomatricula"):
        return "socioeconomica"
    if c.startswith("inst_") or c.startswith("estu_inst_"):
        return "institucion"
    if c.startswith("estu_prgm_") or c in {"estu_nucleo_pregrado", "estu_nivel_prgm_academico", "estu_metodo_prgm", "estu_snies_prgmacademico"}:
        return "programa"
    if c in {"periodo", "estu_consecutivo"}:
        return "identificador/periodo"
    return "personal/geografia/otros"


cols["familia"] = cols.columna.map(familia)
display(cols.groupby("familia").columna.apply(lambda s: ", ".join(s)).to_frame("columnas"))

columnas: 57 | tipos: {'text': 57}
columnas con 'global' en el nombre: []


,columnas
familia,
identificador/periodo,"periodo, estu_consecutivo"
institucion,"inst_cod_institucion, inst_nombre_institucion, inst_caracter_academico, estu_inst_depa..."
personal/geografia/otros,"estu_tipodocumento, estu_pais_reside, estu_cod_reside_depto, estu_depto_reside, estu_c..."
programa,"estu_nucleo_pregrado, estu_prgm_academico, estu_prgm_departamento, estu_prgm_codmunici..."
puntajes/desempenos por modulo (OBJETIVO o FUGA),"mod_razona_cuantitat_punt, mod_comuni_escrita_punt, mod_comuni_escrita_desem, mod_ingl..."
socioeconomica,"estu_valormatriculauniversidad, estu_pagomatriculabeca, estu_pagomatriculacredito, est..."


**[VERIFICADO]** El conjunto unificado **no trae `punt_global`** (solo 5 puntajes por módulo y 2 desempeños). La versión de 2019-2 (`2x55-9wxm`, 105 columnas) sí lo trae. Se verifica abajo cómo se compone para poder reconstruirlo.

## 3. Cómo se compone el puntaje global (2019-2)

Se baja una muestra de 3.000 filas de `2x55-9wxm` (solo 6 columnas de puntajes) y se compara el global con el promedio de los 5 módulos.

In [3]:
def soql(did, params, fmt="json"):
    """Consulta SoQL: timeout 30 s, hasta 3 intentos, se detiene si el portal limita el uso (403/429)."""
    url = f"{BASE}/resource/{did}.{fmt}"
    ultimo = None
    for intento in range(3):
        try:
            r = requests.get(url, params=params, timeout=30)
        except requests.RequestException as e:
            ultimo = str(e)[:80]
            time.sleep(5 * (intento + 1))
            continue
        if r.status_code in (403, 429):
            raise RuntimeError(f"portal limita el uso ({r.status_code}): detenerse")
        if r.status_code == 200:
            if fmt == "json":
                return r.json()
            return pd.read_csv(io.StringIO(r.content.decode("utf-8")), dtype=str, low_memory=False)
        ultimo = f"status {r.status_code}"
        time.sleep(5 * (intento + 1))
    raise RuntimeError(f"fallo {did}: {ultimo}")


f = RAIZ / "global_2019_2_muestra.parquet"
if f.exists():
    g19 = pd.read_parquet(f)
else:
    sel = "punt_global,mod_razona_cuantitat_punt,mod_lectura_critica_punt,mod_competen_ciudada_punt,mod_ingles_punt,mod_comuni_escrita_punt"
    g19 = soql("2x55-9wxm", {"$select": sel, "$limit": 3000}, fmt="csv")
    g19.to_parquet(f)
g19 = g19.apply(pd.to_numeric, errors="coerce").dropna()
mods19 = [c for c in g19.columns if c != "punt_global"]
dif = g19.punt_global - g19[mods19].mean(axis=1)
A = np.c_[g19[mods19].to_numpy(), np.ones(len(g19))]
coef, *_ = np.linalg.lstsq(A, g19.punt_global.to_numpy(), rcond=None)
r2 = 1 - ((g19.punt_global - A @ coef) ** 2).sum() / ((g19.punt_global - g19.punt_global.mean()) ** 2).sum()
print(f"filas: {len(g19)} | diferencia |global - media de los 5 modulos|: media {dif.abs().mean():.3f}, maxima {dif.abs().max():.3f}")
print("coeficientes OLS:", dict(zip(mods19 + ["const"], np.round(coef, 3))), "| R2:", round(r2, 5))

filas: 2997 | diferencia |global - media de los 5 modulos|: media 0.240, maxima 0.400
coeficientes OLS: {'mod_razona_cuantitat_punt': np.float64(0.2), 'mod_lectura_critica_punt': np.float64(0.2), 'mod_competen_ciudada_punt': np.float64(0.2), 'mod_ingles_punt': np.float64(0.2), 'mod_comuni_escrita_punt': np.float64(0.2), 'const': np.float64(0.011)} | R2: 0.99986


**[VERIFICADO en 2019-2]** El puntaje global es la **media simple de los 5 módulos** (cada módulo pesa 0,2; R² = 0,9999; diferencia máxima 0,4 por redondeo). Por eso los puntajes por módulo no pueden ser features si el objetivo es el global: el global es una función casi exacta de ellos. **[SUPUESTO]** la misma fórmula rige en los demás periodos; no se pudo comprobar porque el conjunto unificado no trae el global.

En el baseline, el objetivo es el **global reconstruido** (media de los 5 módulos).

## 4. Muestra de Saber Pro (máx. 50.000 filas)

- 6 periodos grandes: 2018-3, 2019-5, 2020-3, 2021-2, 2022-2 y 2022-5, con **8.000 filas por periodo = 48.000**.
- Aleatoria sin usar `$order + $limit` (sesga hacia el inicio): se filtra por el último dígito de `estu_consecutivo` (con tantos dígitos como hagan falta), se baja ese subconjunto en páginas de 5.000 filas (caché en disco por página) y se submuestrea localmente con semilla fija.
- **No se descargan** `estu_fechanacimiento` ni `estu_privado_libertad` (datos sensibles / cuasi-identificadores). Por eso la **edad no es una feature**.

In [4]:
PERIODOS = ["20183", "20195", "20203", "20212", "20222", "20225"]
CUOTA = 8000
PAGINA = 5000
EXCLUIR_PII = {"estu_fechanacimiento", "estu_privado_libertad"}
SELECT = [c for c in cols.columna if c not in EXCLUIR_PII]
n_pob = per.set_index("periodo").filas.to_dict()


def plan(periodo):
    g = int(np.clip(math.ceil(10 * CUOTA * 1.3 / n_pob[periodo]), 1, 10))
    dig = sorted(np.random.default_rng([SEED, int(periodo)]).choice(10, size=g, replace=False).tolist())
    where = f"periodo = '{periodo}'"
    if g < 10:
        where += " AND (" + " OR ".join(f"estu_consecutivo like '%{d}'" for d in dig) + ")"
    return where, g


def bajar_periodo(periodo, avance):
    where, g = plan(periodo)
    partes, offset = [], 0
    while True:
        f = CACHE / f"{periodo}_{offset:07d}.parquet"
        if f.exists():
            df = pd.read_parquet(f)
        else:
            df = soql(DID, {"$select": ",".join(SELECT), "$where": where, "$order": ":id", "$limit": PAGINA, "$offset": offset}, fmt="csv")
            df.to_parquet(f)
        avance["paginas"] += 1
        if avance["paginas"] % 5 == 0:
            print(f"  progreso: {avance['paginas']} paginas | {time.time() - avance['t0']:.0f} s | periodo {periodo}, filas acumuladas del periodo {offset + len(df)}")
        partes.append(df)
        if len(df) < PAGINA:
            break
        offset += PAGINA
    return pd.concat(partes, ignore_index=True), g


t0 = time.time()
avance = {"paginas": 0, "t0": t0}
trozos = []
for p in PERIODOS:
    df, g = bajar_periodo(p, avance)
    r = np.random.default_rng([SEED, int(p)])
    if len(df) > CUOTA:
        df = df.iloc[np.sort(r.choice(len(df), size=CUOTA, replace=False))]
    trozos.append(df)
    print(f"periodo {p}: subconjunto bajado por {g} digito(s), muestra {len(df)} filas")
print(f"descarga lista: {avance['paginas']} paginas en {time.time() - t0:.0f} s")
sp = pd.concat(trozos, ignore_index=True)
print("filas en la muestra:", len(sp), "| ids duplicados:", int(sp.estu_consecutivo.duplicated().sum()))
sp = sp.drop_duplicates("estu_consecutivo").reset_index(drop=True)
ram("muestra cargada")

  progreso: 5 paginas | 0 s | periodo 20183, filas acumuladas del periodo 23534

periodo 20183: subconjunto bajado por 1 digito(s), muestra 8000 filas


  progreso: 10 paginas | 0 s | periodo 20195, filas acumuladas del periodo 25000
periodo 20195: subconjunto bajado por 1 digito(s), muestra 8000 filas


  progreso: 15 paginas | 0 s | periodo 20203, filas acumuladas del periodo 20000
periodo 20203: subconjunto bajado por 1 digito(s), muestra 8000 filas


  progreso: 20 paginas | 0 s | periodo 20212, filas acumuladas del periodo 20000
periodo 20212: subconjunto bajado por 1 digito(s), muestra 8000 filas


  progreso: 25 paginas | 0 s | periodo 20222, filas acumuladas del periodo 19610
periodo 20222: subconjunto bajado por 2 digito(s), muestra 8000 filas


periodo 20225: subconjunto bajado por 1 digito(s), muestra 8000 filas
descarga lista: 28 paginas en 0 s
filas en la muestra: 48000 | ids duplicados: 126


[RAM] muestra cargada: actual 328 MB | pico 328 MB

In [5]:
MODS = ["mod_razona_cuantitat_punt", "mod_lectura_critica_punt", "mod_competen_ciudada_punt", "mod_ingles_punt", "mod_comuni_escrita_punt"]
for c in MODS:
    sp[c] = pd.to_numeric(sp[c], errors="coerce")
sp["anio"] = sp.periodo.str[:4].astype(int)
print("filas por periodo:", sp.periodo.value_counts().sort_index().to_dict())
print("\nnulos (%) de los puntajes:", (sp[MODS].isna().mean() * 100).round(2).to_dict())
print("ceros (%) de los puntajes:", ((sp[MODS] == 0).mean() * 100).round(2).to_dict())
print("consentimiento para usar los datos en investigacion (estu_estadoinvestigacion):", sp.estu_estadoinvestigacion.value_counts(dropna=False).to_dict())
display(sp[MODS].describe().T.round(1))

filas por periodo: {'20183': 8000, '20195': 8000, '20203': 8000, '20212': 7973, '20222': 8000, '20225': 7901}

nulos (%) de los puntajes: {'mod_razona_cuantitat_punt': 0.0, 'mod_lectura_critica_punt': 0.0, 'mod_competen_ciudada_punt': 0.0, 'mod_ingles_punt': 0.01, 'mod_comuni_escrita_punt': 0.48}
ceros (%) de los puntajes: {'mod_razona_cuantitat_punt': 0.04, 'mod_lectura_critica_punt': 0.14, 'mod_competen_ciudada_punt': 0.28, 'mod_ingles_punt': 0.7, 'mod_comuni_escrita_punt': 5.19}
consentimiento para usar los datos en investigacion (estu_estadoinvestigacion): {'PUBLICAR': 47793, 'PRESENTE CON LECTURA TARDIA': 46, 'VALIDEZ OFICINA JURÍDICA': 30, 'NO SE COMPROBO IDENTIDAD DEL EXAMINADO': 5}


,count,mean,std,min,25%,50%,75%,max
mod_razona_cuantitat_punt,47874.0,146.7,32.1,0.0,123.0,147.0,169.0,300.0
mod_lectura_critica_punt,47874.0,149.8,31.1,0.0,127.0,150.0,172.0,300.0
mod_competen_ciudada_punt,47874.0,145.2,33.8,0.0,121.0,146.0,169.0,300.0
mod_ingles_punt,47869.0,153.8,33.6,0.0,132.0,152.0,175.0,300.0
mod_comuni_escrita_punt,47644.0,137.1,45.0,0.0,122.0,139.0,164.0,300.0


## 5. Fuga de información

Si el objetivo es el puntaje global, **no pueden ser features**:

| Columna(s) | Por qué |
|---|---|
| `mod_razona_cuantitat_punt`, `mod_lectura_critica_punt`, `mod_competen_ciudada_punt`, `mod_ingles_punt`, `mod_comuni_escrita_punt` | Componen el global (media simple, R² = 0,9999): el objetivo es función de ellas. |
| `mod_comuni_escrita_desem`, `mod_ingles_desem` | Niveles de desempeño derivados de los puntajes del propio módulo. |
| `estu_consecutivo`, `periodo` | Identificador del registro / del periodo: no son información del estudiante (el periodo se usa solo para el split). |

Las demás columnas (socioeconómicas, institución, programa, personales) se declaran al inscribirse, antes de la prueba. **[SUPUESTO]** que todas se conocen antes del examen: el cuestionario de contexto se responde al inscribirse, pero algunas respuestas (p. ej. horas de trabajo) podrían actualizarse; no hay forma de comprobarlo.

Aparte de la fuga estricta, hay otro problema: sin el puntaje de Saber 11 (que no se puede cruzar) falta la variable que más predice el desempeño en Saber Pro.

## 6. Baseline

- **Objetivo**: puntaje global reconstruido = media de los 5 módulos (solo filas con los 5 puntajes).
- **Features seguras** (todas categóricas, codificación ordinal; las de alta cardinalidad se limitan a las 200 más frecuentes del train): socioeconómicas, institución, programa y personales/geografía (ver tabla de familias abajo).
- **Split por periodo**: train = 2018-3, 2019-5, 2020-3, 2021-2; test = 2022-2 y 2022-5 (el año más reciente disponible). Para comparar, también un **split aleatorio 80/20** sobre toda la muestra.
- `HistGradientBoostingRegressor` por defecto (`early_stopping=False`; el automático usa un 10 % aleatorio).
- Referencias: predecir la media del train y predecir la media por institución del train.

In [6]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split

sp["y"] = sp[MODS].mean(axis=1, skipna=False)
# conservador: solo registros con estu_estadoinvestigacion = PUBLICAR (se excluyen lectura tardia, validez juridica, identidad no comprobada)
n_antes = len(sp)
d = sp[sp.estu_estadoinvestigacion == "PUBLICAR"].dropna(subset=["y"]).copy()
print("filas en la muestra:", n_antes, "| con estado PUBLICAR y los 5 puntajes:", len(d), f"({len(d) / n_antes:.1%})")

FAMILIAS = {
    "socioeconomica": ["fami_educacionpadre", "fami_educacionmadre", "fami_estratovivienda", "fami_tieneautomovil", "fami_tienelavadora",
                       "fami_tienecomputador", "fami_tieneinternet", "estu_horassemanatrabaja", "estu_valormatriculauniversidad",
                       "estu_pagomatriculabeca", "estu_pagomatriculacredito", "estu_pagomatriculapadres", "estu_pagomatriculapropio"],
    "institucion": ["inst_cod_institucion", "inst_caracter_academico", "inst_origen", "estu_inst_departamento", "estu_inst_municipio"],
    "programa": ["estu_nucleo_pregrado", "estu_prgm_academico", "estu_nivel_prgm_academico", "estu_metodo_prgm", "estu_snies_prgmacademico"],
    "personal_geografia": ["estu_genero", "estu_nacionalidad", "estu_depto_reside", "estu_mcpio_reside", "estu_cole_termino",
                           "estu_coddane_cole_termino", "estu_tipodocumentosb11", "estu_estudiante", "estu_depto_presentacion"],
}
FEATS = [c for fam in FAMILIAS.values() for c in fam]
assert all(c in d.columns for c in FEATS), [c for c in FEATS if c not in d.columns]
print("features:", len(FEATS), "| por familia:", {k: len(v) for k, v in FAMILIAS.items()})
display(pd.DataFrame({"nulos_%": (d[FEATS].isna().mean() * 100).round(1), "n_unicos": d[FEATS].nunique()}).T)


def preparar(tr, te, feats, top=200):
    Xtr, Xte = pd.DataFrame(index=tr.index), pd.DataFrame(index=te.index)
    for c in feats:
        a_, b_ = tr[c].astype("string"), te[c].astype("string")
        frec = a_.value_counts().head(top).index
        a2 = a_.where(a_.isin(frec), "__otros__").fillna("__nan__").to_frame()
        b2 = b_.where(b_.isin(frec), "__otros__").fillna("__nan__").to_frame()
        enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1).fit(a2)
        Xtr[c] = enc.transform(a2)[:, 0]
        Xte[c] = enc.transform(b2)[:, 0]
    Xte = Xte.replace(-1, np.nan)
    return Xtr, Xte


def evaluar(tr, te, nombre):
    Xtr, Xte = preparar(tr, te, FEATS)
    t0 = time.perf_counter()
    m = HistGradientBoostingRegressor(categorical_features=[True] * len(FEATS), early_stopping=False, random_state=SEED).fit(Xtr, tr.y)
    seg = time.perf_counter() - t0
    p_m = m.predict(Xte)
    media_inst = tr.groupby("inst_cod_institucion").y.mean()
    p_i = te.inst_cod_institucion.map(media_inst).fillna(tr.y.mean()).to_numpy()
    res = pd.DataFrame({
        "MAE": [mean_absolute_error(te.y, p_m), mean_absolute_error(te.y, np.full(len(te), tr.y.mean())), mean_absolute_error(te.y, p_i)],
        "R2": [r2_score(te.y, p_m), r2_score(te.y, np.full(len(te), tr.y.mean())), r2_score(te.y, p_i)]},
        index=["HGB (features seguras)", "ref: media del train", "ref: media por institucion"])
    res.insert(0, "split", nombre)
    return res, m, Xte, seg


t_total = time.perf_counter()
tr_p = d[d.periodo.isin(["20183", "20195", "20203", "20212"])]
te_p = d[d.periodo.isin(["20222", "20225"])]
res_p, mod_p, Xte_p, seg_p = evaluar(tr_p, te_p, f"por periodo (train {len(tr_p)}, test {len(te_p)})")
tr_a, te_a = train_test_split(d, test_size=0.2, random_state=SEED)
res_a, mod_a, Xte_a, seg_a = evaluar(tr_a, te_a, f"aleatorio 80/20 (train {len(tr_a)}, test {len(te_a)})")
res = pd.concat([res_p, res_a]).round(3)
display(res)
print(f"desviacion tipica del objetivo en el test por periodo: {te_p.y.std():.1f} | media train {tr_p.y.mean():.1f} vs media test {te_p.y.mean():.1f}")
print(f"entrenamiento: {seg_p:.1f} s y {seg_a:.1f} s | total {time.perf_counter() - t_total:.1f} s")
ram("tras el baseline")

filas en la muestra: 47874 | con estado PUBLICAR y los 5 puntajes: 47560 (99.3%)
features: 32 | por familia: {'socioeconomica': 13, 'institucion': 5, 'programa': 5, 'personal_geografia': 9}


,fami_educacionpadre,fami_educacionmadre,fami_estratovivienda,fami_tieneautomovil,fami_tienelavadora,fami_tienecomputador,fami_tieneinternet,estu_horassemanatrabaja,estu_valormatriculauniversidad,estu_pagomatriculabeca,estu_pagomatriculacredito,estu_pagomatriculapadres,estu_pagomatriculapropio,inst_cod_institucion,inst_caracter_academico,inst_origen,estu_inst_departamento,estu_inst_municipio,estu_nucleo_pregrado,estu_prgm_academico,estu_nivel_prgm_academico,estu_metodo_prgm,estu_snies_prgmacademico,estu_genero,estu_nacionalidad,estu_depto_reside,estu_mcpio_reside,estu_cole_termino,estu_coddane_cole_termino,estu_tipodocumentosb11,estu_estudiante,estu_depto_presentacion
nulos_%,3.4,3.4,4.3,6.1,5.3,5.1,3.9,4.5,0.5,0.5,0.5,0.5,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.2,0.2,30.8,32.4,1.5,0.0,0.0
n_unicos,12.0,12.0,7.0,2.0,2.0,2.0,2.0,5.0,8.0,2.0,2.0,2.0,2.0,262.0,4.0,6.0,26.0,60.0,56.0,884.0,1.0,3.0,3904.0,2.0,22.0,34.0,919.0,9794.0,7068.0,6.0,1.0,33.0


,split,MAE,R2
HGB (features seguras),"por periodo (train 31663, test 15897)",17.230,0.314
ref: media del train,"por periodo (train 31663, test 15897)",21.499,-0.014
ref: media por institucion,"por periodo (train 31663, test 15897)",18.389,0.224
HGB (features seguras),"aleatorio 80/20 (train 38048, test 9512)",15.748,0.378
ref: media del train,"aleatorio 80/20 (train 38048, test 9512)",20.406,-0.000
ref: media por institucion,"aleatorio 80/20 (train 38048, test 9512)",17.086,0.275


desviacion tipica del objetivo en el test por periodo: 26.9 | media train 147.6 vs media test 144.4
entrenamiento: 0.8 s y 1.1 s | total 5.0 s
[RAM] tras el baseline: actual 553 MB | pico 585 MB


**Lectura del baseline** (muestra de 47.560 filas; test = 2022):

| | MAE | R² |
|---|---|---|
| HGB con features seguras (split por periodo) | **17,2** | **0,31** |
| Referencia: media del train | 21,5 | -0,01 |
| Referencia: media por institución | 18,4 | 0,22 |
| HGB, split aleatorio 80/20 (optimista) | 15,7 | 0,38 |

- **[VERIFICADO] Hay señal moderada**: el modelo baja el MAE un 20 % frente a predecir la media y explica ~31 % de la varianza con variables anteriores a la prueba. La desviación típica del puntaje global es ~27 puntos, así que un MAE de 17 sigue siendo grande.
- **[VERIFICADO] Casi todo lo explica la institución**: la media por institución ya logra R² = 0,22; el modelo mejora solo un poco (MAE 18,4 a 17,2).
- **El split aleatorio infla el resultado** (R² 0,38 frente a 0,31 por periodo), porque mezcla periodos e instituciones. Por eso el split principal es por periodo. Solo hay 4 periodos de train y un año de test, así que la estabilidad temporal no se pudo evaluar a fondo.
- La media del puntaje baja un poco entre el train (147,6) y el test (144,4).
- **Sin Saber 11** (no se puede cruzar), falta la variable que más suele predecir el desempeño en Saber Pro; este valor es un piso, no un techo.

## 7. Importancia por permutación (split por periodo)

Se permutan las columnas de a una y también **por familia** (permutando juntas todas las de la familia), sobre el test. Se reporta el aumento del MAE.

In [7]:
rng = np.random.default_rng(SEED)
idx = rng.choice(len(Xte_p), size=min(8000, len(Xte_p)), replace=False)
Xs, ys = Xte_p.iloc[idx].reset_index(drop=True), te_p.y.iloc[idx].reset_index(drop=True)
base = mean_absolute_error(ys, mod_p.predict(Xs))


def aumento(cols_, reps=5):
    v = []
    for k in range(reps):
        X2 = Xs.copy()
        perm = np.random.default_rng([SEED, k]).permutation(len(X2))
        X2[cols_] = Xs[cols_].iloc[perm].to_numpy()
        v.append(mean_absolute_error(ys, mod_p.predict(X2)) - base)
    return float(np.mean(v))


fam_imp = pd.Series({k: aumento(v) for k, v in FAMILIAS.items()}).sort_values(ascending=False)
col_imp = pd.Series({c: aumento([c], reps=3) for c in FEATS}).sort_values(ascending=False)
print(f"MAE de referencia en la submuestra de test: {base:.2f}")
display(fam_imp.round(2).to_frame("aumento_MAE_al_permutar_la_familia"))
display(col_imp.head(12).round(2).to_frame("aumento_MAE_al_permutar_la_columna"))

MAE de referencia en la submuestra de test: 17.17


,aumento_MAE_al_permutar_la_familia
institucion,3.62
programa,0.87
personal_geografia,0.79
socioeconomica,0.72


,aumento_MAE_al_permutar_la_columna
inst_cod_institucion,3.66
estu_prgm_academico,0.58
estu_tipodocumentosb11,0.48
estu_mcpio_reside,0.21
estu_pagomatriculabeca,0.12
fami_educacionpadre,0.09
estu_horassemanatrabaja,0.07
fami_educacionmadre,0.05
estu_genero,0.04
estu_nucleo_pregrado,0.04


**Lectura de la importancia** (aumento del MAE al permutar):

- **[VERIFICADO] Domina la institución**: permutar la familia "institución" sube el MAE en 3,6 puntos (casi todo por `inst_cod_institucion`, 3,7). Le siguen programa (0,9), personales/geografía (0,8) y socioeconómicas (0,7). Entre las columnas sueltas destacan `estu_prgm_academico` (0,6) y `estu_tipodocumentosb11` (0,5).
- **El estrato por sí solo pesa casi nada** (no aparece entre las 12 columnas más importantes). La alerta "solo el estrato domina" **no se cumple**.
- **Alertas éticas, de todos modos**:
  - **[SUPUESTO]** La institución puede ser un proxy del origen socioeconómico (el tipo y carácter de la institución se asocian al nivel de ingreso); no se midió aquí. Un modelo que predice el puntaje individual a partir de institución, municipio de residencia y nivel educativo de los padres puede reproducir desigualdad y, si se despliega, etiquetar a una persona por su origen.
  - `estu_tipodocumentosb11` (tipo de documento con el que presentó Saber 11, p. ej. tarjeta de identidad o cédula) probablemente sea un proxy de la **edad**, que se excluyó a propósito por ser dato sensible. Hay que revisarlo.
  - Hay que evaluar el desempeño por grupo (género, estrato, tipo de institución) antes de cualquier uso.

## Costo, actualización y despliegue

- **Costo medido (muestra de 47.560 filas)**: descarga de 28 páginas en ~2 min; cargar y entrenar el HGB ~1 s; todo el notebook ~25 s con la caché y **pico de RAM ~585 MB**.
- **Costo estimado del dataset completo (1,2 M de filas, 57 columnas de texto) [ESTIMADO]**: ~244 páginas de 5.000 filas a ~4,4 s cada una = **~18 min** de descarga; en RAM, unos 3,5 GB como texto (la muestra ocupa ~140 MB por 48 mil filas), bastante menos con columnas categóricas. Cabe en un laptop de 8 GB si se tipan las columnas.
- **Actualización para el reajuste**: datos.gov.co se actualizó por última vez el **2023-08-23** (hasta 2022). No hay 2023-2025 por API. El portal del ICFES no se pudo consultar (su `robots.txt` lo impide) y no se verificó el acceso. **El reajuste automático no es viable con esta fuente**, salvo que el ICFES entregue los años nuevos.
- **Despliegue**: la interfaz para no técnicos sería una calculadora de "puntaje esperado" para un estudiante; con las alertas éticas de arriba es el caso de uso más delicado de los tres candidatos.
- **Licencia**: CC BY-SA 4.0: hay que citar al ICFES y compartir los derivados con la misma licencia.
- **Limitaciones**: el objetivo es un global **reconstruido** (media de 5 módulos, verificada solo en 2019-2); solo 6 de 13 periodos; 8.000 filas por periodo (muestra); sin edad; categóricas de alta cardinalidad recortadas a 200; sin ajuste de hiperparámetros.

## 8. Resumen para el checkpoint

In [8]:
print("Disponibilidad: Saber Pro en datos.gov.co periodos 2018-3 a 2022-6 (1.217.482 filas, 57 columnas); sin 2023-2025")
print("Cruce con Saber 11: no hay llave comun (EK... vs SB11...)")
print("Objetivo: global reconstruido = media simple de 5 modulos (verificado en 2019-2)")
display(res)
display(fam_imp.round(2).to_frame("aumento_MAE"))
ram("final")

Disponibilidad: Saber Pro en datos.gov.co periodos 2018-3 a 2022-6 (1.217.482 filas, 57 columnas); sin 2023-2025
Cruce con Saber 11: no hay llave comun (EK... vs SB11...)
Objetivo: global reconstruido = media simple de 5 modulos (verificado en 2019-2)


,split,MAE,R2
HGB (features seguras),"por periodo (train 31663, test 15897)",17.230,0.314
ref: media del train,"por periodo (train 31663, test 15897)",21.499,-0.014
ref: media por institucion,"por periodo (train 31663, test 15897)",18.389,0.224
HGB (features seguras),"aleatorio 80/20 (train 38048, test 9512)",15.748,0.378
ref: media del train,"aleatorio 80/20 (train 38048, test 9512)",20.406,-0.000
ref: media por institucion,"aleatorio 80/20 (train 38048, test 9512)",17.086,0.275


,aumento_MAE
institucion,3.62
programa,0.87
personal_geografia,0.79
socioeconomica,0.72


[RAM] final: actual 557 MB | pico 585 MB
